In [1]:
import torch
import time
import copy
import os
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

if os.path.basename(os.getcwd()) == 'exercises':
    os.chdir('../')
    
from ctm.ctm import ContinuousThoughtMachine
from ctm.img_coder import MinesweeperConvEncoder

/home/mwuerstle@corp.exxcellent.de/miniconda3/envs/agi-project/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device_config = -1

# Configure device string (support MPS on macOS)
if device_config != -1:
    device = f'cuda:{device_config}'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'Running model CTM on {device}')
torch.set_default_device(device)

Running model CTM on cpu


In [3]:
from environments.minesweeper.minesweeper import MinesweeperEnv

width = 6
height = 6
n_mines = 10

env = MinesweeperEnv(width, height, n_mines)

current_state = env.state_im
board = current_state.reshape(1, env.ntiles)
unsolved = [i for i, x in enumerate(board[0]) if x==-0.125]
move = np.random.choice(unsolved)
env.step(move)
env.draw_state(env.state_im)
print(current_state.shape)



/home/mwuerstle@corp.exxcellent.de/Uni/agi_project/environments/minesweeper/minesweeper.py:123: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(state_df.style.applymap(self.color_state))


,0,1,2,3,4,5
0,-1,-1,-1,1,-1,-1
1,-1,-1,-1,-1,-1,-1
2,-1,-1,-1,-1,-1,-1
3,-1,-1,-1,-1,-1,-1
4,-1,-1,-1,-1,-1,-1
5,-1,-1,-1,-1,-1,-1


(6, 6, 1)


In [ ]:
ctm_latent_dim = 256

env.reset()

minesweeper_enc = MinesweeperConvEncoder(ctm_latent_dim, env.state_im.shape)
# Create Tensor with the correct shape for encoding 
state_tensor = torch.from_numpy(env.state_im.T).float().unsqueeze(0)
encoded_state = minesweeper_enc.forward(state_tensor)
print(encoded_state.shape)

torch.Size([1, 256])


In [11]:
d_input = 256

ctm = ContinuousThoughtMachine(iterations=75, 
                               d_model=2048, 
                               d_input=d_input, 
                               heads=8, 
                               n_synch_out=64, 
                               n_synch_action=32,
                               synapse_depth=8, 
                               memory_length=25, 
                               deep_nlms=False,
                               memory_hidden_dims=32, 
                               do_layernorm_nlm=False,
                               backbone_type='none',
                               positional_embedding_type='none',
                               out_dims=256
                               )

try:
    # Determine pseudo input shape based on dataset
    pseudo_inputs = torch.zeros((1, 1, d_input), device=device).float()
    ctm(pseudo_inputs)
except Exception as e:
        print(f"Warning: Pseudo forward pass failed: {e}")

latent_state = encoded_state.unsqueeze(0)
print(latent_state.shape)
predictions_raw, certainties, synchronisation = ctm(latent_state)
final_thought = predictions_raw[:, :, -1]
print(final_thought.shape)

Using neuron select type: random-pairing
Synch representation size action: 32
Synch representation size out: 64
torch.Size([1, 1, 256])
torch.Size([1, 256])


In [ ]:
def get_action(self, state, env):
    board = state.reshape(1, env.ntiles)
    unsolved = [i for i, x in enumerate(board[0]) if x==-0.125]

    rand = np.random.random() # random value b/w 0 & 1

    if rand < self.epsilon: # random move (explore)
        move = np.random.choice(unsolved)
    else:
        moves = self.model.predict(np.reshape(state, (1, self.env.nrows, self.env.ncols, 1)))
        moves[board!=-0.125] = np.min(moves) # set already clicked tiles to min value
        move = np.argmax(moves)

    return move